# BrainSense Export Starter
**Two output streams per trial:**
- `_td.csv` -- raw time-domain voltage at 250 Hz (`time_s`, `left_uV`, `right_uV`)
- `_lfp.csv` -- device-computed LFP power + stim current at 2 Hz (`time_s`, `left_lfp`, `left_stim_mA`, `right_lfp`, `right_stim_mA`)
- `.pkl` -- full bundle with both streams + metadata for downstream analysis

In [ ]:
import sys
sys.path.insert(0, '..')  # adjust if pypercept is not pip-installed

from pypercept.io import load_session, export_brainsense
from pathlib import Path

## 1. Point to your JSON

In [ ]:
json_path = '/path/to/Report_BaselineEEG_Session_Report.json' 
out_dir   = './export'  #output directory for CSVs & pkls

In [ ]:
session = load_session(json_path)
print(session.summary())

In [ ]:
for sess in session.get_sessions():
    td_l, td_r   = sess['td']['left'],  sess['td']['right']
    lfp_l, lfp_r = sess['lfp']['left'], sess['lfp']['right']
    print(f"Trial {sess['trial_number']}  ({sess['datetime']})")
    if td_l:
        print(f"  TD  left:  {td_l.n_samples:,} samples @ {td_l.sample_rate} Hz  "
              f"({td_l.duration_seconds:.1f}s)  ch={td_l.channel}")
    if td_r:
        print(f"  TD  right: {td_r.n_samples:,} samples @ {td_r.sample_rate} Hz  "
              f"({td_r.duration_seconds:.1f}s)  ch={td_r.channel}")
    if lfp_l:
        print(f"  LFP left:  {lfp_l.n_samples} samples @ {lfp_l.sample_rate} Hz  "
              f"stim={lfp_l.stim_amplitude[0]:.1f}->{lfp_l.stim_amplitude[-1]:.1f} mA")
    if lfp_r:
        print(f"  LFP right: {lfp_r.n_samples} samples @ {lfp_r.sample_rate} Hz  "
              f"stim={lfp_r.stim_amplitude[0]:.1f}->{lfp_r.stim_amplitude[-1]:.1f} mA")
    print()

In [ ]:
written = export_brainsense(json_path, outdir=out_dir)
for p in written:
    print(p)

In [ ]:
import pandas as pd

td_csvs  = [p for p in written if p.name.endswith('_td.csv')]
lfp_csvs = [p for p in written if p.name.endswith('_lfp.csv')]

print('--- TD (250 Hz raw waveform) ---')
df_td = pd.read_csv(td_csvs[0])
print(f'{len(df_td):,} rows, columns: {list(df_td.columns)}')
display(df_td.head())

print('\n--- LFP + stim (2 Hz) ---')
df_lfp = pd.read_csv(lfp_csvs[0])
print(f'{len(df_lfp):,} rows, columns: {list(df_lfp.columns)}')
display(df_lfp.head())

## plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Top: raw TD waveform
ax = axes[0]
ax.plot(df_td['time_s'], df_td['left_uV'],  lw=0.3, alpha=0.7, label='Left')
ax.plot(df_td['time_s'], df_td['right_uV'], lw=0.3, alpha=0.7, label='Right')
ax.set_ylabel('Voltage (uV)')
ax.set_title('BrainSense TimeDomain (250 Hz)')
ax.legend(loc='upper right')

# Bottom: LFP power + stim
ax2 = axes[1]
ax2.plot(df_lfp['time_s'], df_lfp['left_lfp'],  'b-', lw=1, label='Left LFP')
ax2.plot(df_lfp['time_s'], df_lfp['right_lfp'], 'r-', lw=1, label='Right LFP')
ax2.set_ylabel('LFP power (device units)')
ax2.set_xlabel('Time (s)')

ax3 = ax2.twinx()
ax3.plot(df_lfp['time_s'], df_lfp['left_stim_mA'],  'b--', lw=1, alpha=0.5, label='Left stim')
ax3.plot(df_lfp['time_s'], df_lfp['right_stim_mA'], 'r--', lw=1, alpha=0.5, label='Right stim')
ax3.set_ylabel('Stim (mA)')

ax2.set_title('BrainSense LFP + Stim (2 Hz)')
lines2, labels2 = ax2.get_legend_handles_labels()
lines3, labels3 = ax3.get_legend_handles_labels()
ax2.legend(lines2 + lines3, labels2 + labels3, loc='upper right')

plt.tight_layout()
plt.show()

Batch export

In [ ]:
import os

patient_dir = '/path/to/PCT702'  # <-- EDIT THIS
batch_out   = './export_batch'

for root, dirs, files in os.walk(patient_dir):
    for fname in sorted(files):
        if not fname.lower().endswith('.json'):
            continue
        fp = os.path.join(root, fname)
        try:
            s = load_session(fp)
        except Exception:
            continue
        if not s.has_brainsense_timedomain():
            continue  # skip Events-only files
        print(f'Exporting {fname}...')
        export_brainsense(fp, outdir=batch_out)